# 1. Imports

In [ ]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

# 2. Funções

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [ ]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

# 3. Preparação dos dados

## 3.1. Criação da Sessão Spark

In [ ]:
# Criação da sessão Spark local
spark = SparkSession.builder.master("local[*]").appName("feature_engineering").getOrCreate()

## 3.2. Criação do df para pegar os eventos de todas as partidas da temporada de 2022-2023 da Premier League

(dps pode ser interessante levar a parte do schema dos jogadores e da bola p etapa de extração)

In [ ]:
# competition_id = 1 (Premier League)
# season = 2022-2023
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

In [ ]:
#df_events.select('homePlayers', 'awayPlayers', 'balls').first()

In [ ]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        #StructField("speed", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        #StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("z", FloatType(), True),
        StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df_events = df_events.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": F.from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": F.from_json("balls", balls_schema),

    "details_parsed": F.from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

df_events = df_events.select(
    'competitionId',
    'season', # dps mudar pra seasonId se necessário
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    #'periodDescription',
    'startFormattedGameClock',
    'startGameClock',
    #'details_parsed',
    'homeTeam',
    F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    'homePlayers_parsed', 
    'awayPlayers_parsed', 
    'balls_parsed'
)

In [ ]:
#df_events.filter(F.size("homePlayers_parsed") == 0).show()

In [ ]:
#df_events.filter(F.size("awayPlayers_parsed") == 0).show()

In [ ]:
#df_events.filter(F.size("balls_parsed") == 0).show()

In [ ]:
print('Quantidade de linhas:', df_events.count())

## 3.2. Obter jogos da temporada e ajustar identificação do mandante/adversário

(dps pode ser interessante levar isso p etapa de extração)

In [ ]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_raw = df_games.withColumnRenamed("id","gameId").filter(F.col('season') == '2022-2023')

#df_games_raw.show()

In [ ]:
df_games_raw.select('venueType').distinct().show()

In [ ]:
df_games_raw.filter(F.col('venueType') == 'NEUTRAL').show()

- Apesar de Goodison Park teoricamente ser estádio do Everton e se tratarem de todos os jogos do Everton, o venueType é neutro. Por conta disso, não irei considerar essas partidas.
- Caso necessário usá-las futuramente, podemos ver como ficam os eventos de home e away e se seguem essa estrutura acima mesmo o estádio sendo neutro. Uma sugestão pode ser considerar sempre Everton como casa, mas teria que ver se os eventos de posse ficam de acordo.

In [ ]:
df_games_raw = df_games_raw.filter(F.col('venueType').isin(['TEAM_HOME', 'OPPONENT_HOME']))

In [ ]:
# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)
df_games = (
    df_games_raw
    .withColumns({
        "homeTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("`team.id`")).otherwise(F.col("`opponentTeam.id`")),
        "homeTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("`team.name`")).otherwise(F.col("`opponentTeam.name`")),
        
        "opponentTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("`opponentTeam.id`")).otherwise(F.col("`team.id`")),
        "opponentTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("`opponentTeam.name`")).otherwise(F.col("`team.name`")),
    })
    .select(
        'gameId', 
        'date',
        'season',
        F.col('`competition.id`').alias('competitionId'),
        F.col('`competition.name`').alias('competitionName'),
        'venueType',
        'homeTeamId',
        'homeTeamName',
        'opponentTeamId',
        'opponentTeamName',
        F.col('teamStartSide').alias('homeTeamStartSide'),
        F.col('`stadium.name`').alias('stadiumName'), 
        F.col('`stadium.length`').cast("float").alias('stadiumLength'), 
        F.col('`stadium.width`').cast("float").alias('stadiumWidth')
    )
)

# df_games.show(5)

## 4. Junção dos dados dos Jogos + Eventos em uma tabela

In [ ]:
# left join dos eventos + informações dos jogos
df_games_events = (
    df_events.join(
        df_games.drop('season', 'competitionId', 'competitionName'), 
        on = "gameId", 
        how='left'
    )
)

df_games_events = (
    df_games_events
    .withColumn(
        'homeTeamAttackDirection',
            F.when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Right')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Left')), 
                'Left'
            )
            .when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Left')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Right')), 
                'Right'
            )
        )
    .withColumn(
            'awayTeamAttackDirection',
            F.when(F.col('homeTeamAttackDirection') == 'Right', 'Left')
            .when(F.col('homeTeamAttackDirection') == 'Left', 'Right')
        )
    .drop('homeTeamStartSide')
)

df_games_events.show(5)

In [ ]:
df_games_events.filter(F.col('homeTeamName') == F.col('opponentTeamName')).show()

In [ ]:
df_games_events.filter(F.col('homeTeamName') == F.col('opponentTeamName')).select('venueType').distinct().show()

In [ ]:
df_games_events.groupby('period').count().show()

In [ ]:
# variável que indica o time com a posse
df_games_events.groupBy('homeTeam').count().show()

In [ ]:
# df_events.filter(
#     (F.size("balls_parsed") == 0) |
#     (F.size("homePlayers_parsed") == 0) |
#     (F.size("awayPlayers_parsed") == 0)
#     ).show()

In [ ]:
# window function pra criação do Id de posse
w_pos = (
    Window
    .partitionBy(
        "competitionId",
        "season",
        "gameId"
    )
    .orderBy("startGameClock")
)

df_games_events = (
    df_games_events
    .filter(
        # filtro para garantir apenas eventos das partidas no 1º e 2º tempo
        (F.col('period').isin([1,2])) &
        # filtro para não considerar tracking da bola e jogadores home/away que n tem tracking (dps podemos pensar em imputar)
        (F.size("balls_parsed") != 0) & (F.size("homePlayers_parsed") != 0) & (F.size("awayPlayers_parsed") != 0)
    ) 
    .dropna(subset='homeTeam') # drop nos eventos onde nenhum dos dois times tem a posse
    .withColumn(
        "possession_id",
        F.sum(
            F.when(
                F.col("homeTeam") != F.lag("homeTeam").over(w_pos), 1
            ).otherwise(0)
        ).over(
            w_pos.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
)

## Normalização do ataque sempre pra direita

In [ ]:
# time com a posse está atacando e time sem está defendendo

df_games_events_tracking = (
    df_games_events
    # time atacando = se o time da casa tiver a posse, pega tracking home, se não pega tracking away
    .withColumn(
        "attackingPlayers",
        F.when(F.col("homeTeam"), F.col("homePlayers_parsed"))
        .otherwise(F.col("awayPlayers_parsed"))
    )
    # time defendendo = se o time da casa tiver a posse, pega tracking away, se não pega tracking home
    .withColumn(
        "defendingPlayers",
        F.when(F.col("homeTeam"), F.col("awayPlayers_parsed"))
        .otherwise(F.col("homePlayers_parsed"))
    )
    .withColumn(
        "attackingDirection",
        F.when(F.col("homeTeam"), F.col("homeTeamAttackDirection"))
        .otherwise(F.col("awayTeamAttackDirection"))
    )
    # flag para normalização (para tratar ataque sempre pra direita)
    .withColumn(
        "need_side_revert",
        F.col("attackingDirection") == "Left"
    )
    .drop(
        'homePlayers_parsed',  
        'awayPlayers_parsed',
        'homeTeamAttackDirection',
        'awayTeamAttackDirection'
    )
)

#df_games_events_tracking.show()

In [ ]:
players_tracking_norm = (
        lambda p: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -p["x"])
            .otherwise(p["x"])
            .alias("x"),
            # variáveis restantes mantém igual
            p["y"].alias("y"),            
            p["player"].alias("player"),
            p["visibility"].alias("visibility"),
            p["confidence"].alias("confidence")
        )
)

balls_norm = (
    F.transform(
        "balls_parsed",
        lambda b: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -b["x"])
            .otherwise(b["x"])
            .alias("x"),
            # variáveis restantes mantém igual            
            b["y"].alias("y"),
            b["z"].alias("z"),            
            b["visibility"].alias("visibility")
        )
    )
)

# df para normalizar os atacantes, defensores e bola sempre atacando do lado direito
df_games_events_tracking_norm = (
    df_games_events_tracking
    # normalizar atacantes
    .withColumns({
        "attackingPlayersNorm": F.transform("attackingPlayers", players_tracking_norm),
        # normalizar defensores
        "defendingPlayersNorm": F.transform("defendingPlayers", players_tracking_norm),
        # normalizar a bola
        "ballsNorm": balls_norm
    }).drop(
        'attackingPlayers', 
        'defendingPlayers',
        'balls_parsed',
        'attackingDirection',
        'need_side_revert'
        )
)

In [ ]:
def euclidean_dist(x1, y1, x2, y2):
    return F.sqrt(
        F.pow(x1 - x2, 2) +
        F.pow(y1 - y2, 2)
    )

In [ ]:
# Extremos esquerdo e direito do campo no plano cartesiano
left_x = -F.col('stadiumLength') / 2
right_x = F.col('stadiumLength') / 2

# Extremos superior e inferior do campo no plano cartesiano
top_y = F.col('stadiumWidth') / 2
bottom_y = -F.col('stadiumWidth') / 2

# Eixos x e y da posição da bola a partir dos dados de tracking
ball_x = F.get("ballsNorm", 0)["x"]
ball_y = F.get("ballsNorm", 0)["y"]

# Menor distância euclidiana entre os escanteios do lado esquerdo até a bola
progression_distance = F.round(
    F.least(
        euclidean_dist(ball_x, ball_y, left_x, top_y),
        euclidean_dist(ball_x, ball_y, left_x, bottom_y)
    ), 2)

## Criação das componentes de ameaça

- Métrica 1: Distância percorrida no campo (medida pela menor distância entre os escanteios do time com a posse até a bola) 
    - Hipótese: Quanto mais o jogador com posse percorrer o campo com a bola na direção do gol, maior a ameaça de gol por estar mais próximo dele.
    - Relação: Diretamente proporcional
- Métrica 2: Quantidade total de jogadores dos dois times (entre a bola e o gol)
    - Hipótese: Quanto MAIS jogadores dos dois times entre a bola e gol, MENOR a ameaça de gol por haver maior possibilidade de alguma ação defensiva e também por haver chances de um possível chute ser bloqueado.
    - Relação: Inversamente proporcional
- Métrica 3: Vantagem numérica do ataque em relação à defesa (entre a bola e o gol)
    - Hipótese: Quanto MAIOR a vantagem numérica do ataque em relação à defesa, MAIOR a ameaça de gol por ter maiores chance de ações ofensivas e menores chances de ações defensivas
    - Relação: Diretamente proporcional

In [ ]:
df_games_events_players_ball_goal = (
    df_games_events_tracking_norm
     .withColumns({
        # Quantidade de jogadores do time mandante entre o gol esquerdo e a bola
        'attackers_between_ball_goal': (
            F.size(
                F.filter(
                    F.col('attackingPlayersNorm'),
                    lambda p: (
                        # jogadores no eixo x entre a bola e o gol direito
                        (p['x'] <= right_x) &
                        (p['x'] >= ball_x)
                    )
                )
            )
        ),
        'defenders_between_ball_goal': (
            F.size(
                F.filter(
                    F.col('defendingPlayersNorm'),
                    lambda p: (
                        # jogadores no eixo x entre a bola e o gol direito
                        (p['x'] <= right_x) &
                        (p['x'] >= ball_x)
                    )
                )
            )
        )
    })
)

df_games_events_players_ball_goal = (
    df_games_events_players_ball_goal.withColumns({
        # Progressão em campo em direção ao gol defendido
        'progression_distance': progression_distance,
        
        # Quantidade absoluta de jogadores entre a bola e o gol
        'total_players_between_ball_goal': F.col('attackers_between_ball_goal') + F.col('defenders_between_ball_goal'),
        
        # Vantagem numérica do ataque em relação à defesa
        'atk_def_advantage_between_ball_goal': F.col('attackers_between_ball_goal') - F.col('defenders_between_ball_goal')
    })
)

# df_games_events_players_ball_goal.show()

### Criação da Ameaça pela Média das 3 componentes normalizadas com Min-Max

In [ ]:
# Obtém mínimos e máximos das componentes
stats = (
    df_games_events_players_ball_goal
    .agg(
        F.min("progression_distance").alias("min_pd"),
        F.max("progression_distance").alias("max_pd"),

        F.min("total_players_between_ball_goal").alias("min_tp"),
        F.max("total_players_between_ball_goal").alias("max_tp"),

        F.min("atk_def_advantage_between_ball_goal").alias("min_adv"),
        F.max("atk_def_advantage_between_ball_goal").alias("max_adv")
    )
    .first()
)

df_games_events_players_ball_goal_norm = (
    df_games_events_players_ball_goal
    # Componentes normalizadas [0,1]
    .withColumns({
        # Progressão em campo em direção ao gol defendido
        "progression_distance_norm":
        F.round((F.col("progression_distance") - F.lit(stats["min_pd"])) / F.lit(stats["max_pd"] - stats["min_pd"]), 3),

        # Quantidade absoluta de jogadores entre a bola e o gol (1 - minmax por ser inversamente proporcional)
        "total_players_between_ball_goal_norm": F.round(1 - 
        (F.col("total_players_between_ball_goal") - F.lit(stats["min_tp"])) / F.lit(stats["max_tp"] - stats["min_tp"]), 3), 
        
        # Vantagem numérica do ataque em relação à defesa
        "atk_def_advantage_between_ball_goal_norm":
        F.round((F.col("atk_def_advantage_between_ball_goal") - F.lit(stats["min_adv"])) / F.lit(stats["max_adv"] - stats["min_adv"]), 3),

        # Threat score = média das 3 componentes
        "threat_score": F.round((
            F.col("progression_distance_norm") + F.col("total_players_between_ball_goal_norm") + F.col("atk_def_advantage_between_ball_goal_norm")
        ) / F.lit(3.0), 3)

    })
)

In [ ]:
# window function por competição-temporada-jogo-time com posse
w = (
    Window
    .partitionBy(
        "competitionId",
        "season",
        "gameId",
        "homeTeam"
    )
    .orderBy("startGameClock")
)

df_threat_final = (
    df_games_events_players_ball_goal_norm
    .withColumn(
        "threat_score_delta",
        F.round(F.coalesce(F.col("threat_score") - F.lag("threat_score").over(w), F.lit(0.0)), 3))
)

### Validação do threat_score criado

In [ ]:
filtered_columns = [
    'gameId',
    'competitionId',
    'season',
    'date',
    'eventId',
    'eventType',
    'period',
    'startFormattedGameClock',
    'startGameClock',
    'homeTeam',
    'homeTeamName',
    'opponentTeamName',
    'possession_id',
    'threat_score'
]

df_threat_filtrado = (
    df_threat_final.select(filtered_columns)
)

df_threat_filtrado.cache()
df_threat_filtrado.show()

In [ ]:
pl_match_stats_22_23 = str(Path().resolve().parent.parent / "data" / "match_stats" / "PL_22_23.csv")

df_pl_match_stats_22_23 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(pl_match_stats_22_23, sep=',')    

df_pl_match_stats_22_23.show()

In [ ]:
df_pl_match_stats_22_23_filtrado = df_pl_match_stats_22_23.select(
    F.to_date(F.col("Date"), "dd/MM/yyyy").alias("date"),
    F.col('HomeTeam').alias('homeTeamName'),
    F.col('AwayTeam').alias('opponentTeamName'),
    'FTHG',
    'FTAG',
    'FTR',
    'HTHG',
    'HTAG',
    'HTR',
    'HS',
    'AS',
    'HST',
    'AST',    
    'AvgH',
    'AvgA',
    'AvgD'
).sort('date')

In [ ]:
team_name_mapping = {
    "Tottenham": "Tottenham Hotspur",
    "Brighton": "Brighton & Hove Albion",
    "Man City": "Manchester City",
    "Crystal Palace": "Crystal Palace",
    "Leicester": "Leicester City",
    "Aston Villa": "Aston Villa",
    "Bournemouth": "AFC Bournemouth",
    "Fulham": "Fulham",
    "West Ham": "West Ham",
    "Man United": "Manchester United",
    "Wolves": "Wolverhampton Wanderers",
    "Southampton": "Southampton",
    "Liverpool": "Liverpool",
    "Chelsea": "Chelsea",
    "Nott'm Forest": "Nottingham Forest",
    "Newcastle": "Newcastle United",
    "Everton": "Everton",
    "Leeds": "Leeds United",
    "Arsenal": "Arsenal",
    "Brentford": "Brentford"
}

df_pl_match_stats_22_23_filtrado_mapped = (
    df_pl_match_stats_22_23_filtrado
    .replace(team_name_mapping, subset=["homeTeamName", "opponentTeamName"])
)

df_pl_match_stats_22_23_filtrado_mapped.show()

In [ ]:
# df_threat_filtrado.select('homeTeamName').distinct().show(truncate=False)
# df_pl_match_stats_22_23.select('homeTeamName').distinct().show(truncate=False)

In [ ]:
df_threat_filtrado_match_stats = (
    df_threat_filtrado.join(
        df_pl_match_stats_22_23,
        on= ['date', 'homeTeamName', 'opponentTeamName'],
        how='left'
    )
)

df_threat_filtrado_match_stats.show()

In [ ]:
df_threat_filtrado_match_stats.filter(F.col('Div').isNull()).show()

In [ ]:
df_threat_filtrado_match_stats.filter(F.col('Div').isNull()).select('gameId').distinct().show()

In [ ]:
df_games_raw.filter(F.col('gameId') == 4458).show()
df_games.filter(F.col('gameId') == 4458).show()
df_threat_filtrado.filter(F.col('gameId') == 4458).show()

In [ ]:
df_games_events_players_ball_goal_norm.filter(F.col('gameId') == 4458).show()

In [ ]:
df_events.filter(F.col('gameId') == 4458).show()